In [190]:
from nba_api.stats.endpoints import playerindex, playbyplayv3 


In [192]:
from nba_api.stats.endpoints import LeagueGameFinder
import pandas as pd

# מציאת כל המשחקים מהעונה הסדירה 2014-15
game_finder = LeagueGameFinder(season_nullable='2014-15', league_id_nullable='00')
games = game_finder.get_data_frames()[0]

# הצגת המשחקים הראשונים בעונה (כולל GameID)
games_df = games[['GAME_ID', 'GAME_DATE', 'MATCHUP']]
print(games_df.head(10))


      GAME_ID   GAME_DATE      MATCHUP
0  0041400406  2015-06-16  CLE vs. GSW
1  0041400406  2015-06-16    GSW @ CLE
2  0041400405  2015-06-14    CLE @ GSW
3  0041400405  2015-06-14  GSW vs. CLE
4  0041400404  2015-06-11  CLE vs. GSW
5  0041400404  2015-06-11    GSW @ CLE
6  0041400403  2015-06-09  CLE vs. GSW
7  0041400403  2015-06-09    GSW @ CLE
8  0041400402  2015-06-07  GSW vs. CLE
9  0041400402  2015-06-07    CLE @ GSW


In [6]:
from nba_api.stats.endpoints import PlayByPlayV2

# בחר GameID של משחק מעונת 2014-15
game_id = '0021400001'  # לדוגמה, משחק פתיחת העונה

# שליפת נתוני Play-by-Play
pbp = PlayByPlayV2(game_id=game_id)
pbp_df = pbp.get_data_frames()[0]

# הצגת הפעולות הראשונות במשחק
print(pbp_df.head(10))


      GAME_ID  EVENTNUM  EVENTMSGTYPE  EVENTMSGACTIONTYPE  PERIOD  \
0  0021400001         0            12                   0       1   
1  0021400001         1            10                   0       1   
2  0021400001         2             2                   1       1   
3  0021400001         3             4                   0       1   
4  0021400001         4             1                  66       1   
5  0021400001         5             5                   1       1   
6  0021400001         6             5                   4       1   
7  0021400001         7             2                   1       1   
8  0021400001         8             4                   0       1   
9  0021400001         9             2                   5       1   

  WCTIMESTRING PCTIMESTRING                              HOMEDESCRIPTION  \
0      8:13 PM        12:00                                         None   
1      8:14 PM        12:00  Jump Ball Davis vs. Vucevic: Tip to Holiday   
2      8:14 

In [194]:
def convert_time_to_seconds(time_str):
    """ ממיר זמן בפורמט MM:SS לשניות """
    minutes, seconds = map(int, time_str.split(":"))
    return minutes * 60 + seconds

def compute_absolute_time(df):
    """ מחשב זמן מוחלט למשחק על בסיס PERIOD ו-PCTIMESTRING """
    df = df.copy()
    
    # המרת הזמן הנותר ברבע לשניות
    df['SECONDS_LEFT_IN_Q'] = df['PCTIMESTRING'].apply(convert_time_to_seconds)
    
    # חישוב הזמן המוחלט (כל רבע 720 שניות, מחצית 1440, וכו')
    df['ABS_TIME'] = (df['PERIOD'] - 1) * 720 + (720 - df['SECONDS_LEFT_IN_Q'])
    
    return df

# הפעלת הפונקציה על ה- DataFrame שלך
pbp_df = compute_absolute_time(pbp_df)


In [196]:
def calculate_minutes_played(play_by_play_df, until_clock):
    """
    Calculates the number of minutes played by each player until the given clock timestamp.
    """
    # Parse until_clock into a timedelta for comparison
    #until_time = Timedelta(until_clock, unit='s')

    # Sort the data by period and actionNumber to simulate the game progression
    play_by_play_df = play_by_play_df.sort_values(by=['PERIOD', 'EVENTNUM']).reset_index(drop=True)

    # Track active players on the court for each team
    active_players = {}
    total_play_time = {}

    # Filter out substitution events
    sub_events = play_by_play_df[(play_by_play_df['EVENTMSGTYPE'] == 8) & (play_by_play_df['ABS_TIME'] <= until_clock)].reset_index(drop=True)

    # סינון חילופים עד הזמן הרצוי
    sub_events = play_by_play_df[(play_by_play_df['EVENTMSGTYPE'] == 8) & (play_by_play_df['ABS_TIME'] <= until_clock)].reset_index(drop=True)

    # 🔹 הוספת חמישיית הפתיחה
    starting_players = set()

    for _, row in play_by_play_df.iterrows():
        if row['EVENTMSGTYPE'] == 8:  # עצירה כאשר מגיעים לחילוף הראשון
            break
        if pd.notna(row['PLAYER1_NAME']):  
            starting_players.add((row['PLAYER1_NAME'], row.get('PLAYER1_TEAM_ABBREVIATION', 'Unknown')))

    # הוספת שחקני הפתיחה עם זמן 0 (הם התחילו לשחק מהשנייה הראשונה של המשחק)
    for player, team in starting_players:
        if team not in active_players:
            active_players[team] = {}
        active_players[team][player] = 0  # שחקן שהתחיל את המשחק התחיל לשחק מ-0 שניות

    print("🏀 חמישייה פותחת שהוספנו:", starting_players)


    for _, row in sub_events.iterrows():
        current_time = row['ABS_TIME']
        print(current_time)

        player_in = row['PLAYER2_NAME']  # השחקן שנכנס
        player_out = row['PLAYER1_NAME']  # השחקן שיצא


        # Handle substitution logic for active players
        team = row.get('PLAYER1_TEAM_ABBREVIATION', 'Unknown')  # קוד הקבוצה (מתוקן)
        print(f"🔄 חילוף: {player_out} → {player_in} בקבוצה {team} בזמן {current_time}")  # ✅ הדפס חילוף

        if team not in active_players:
            active_players[team] = {}

        # Record play time for the player being substituted out
        if player_out in active_players[team]:
            play_start_time = active_players[team].pop(player_out)
            minutes_played = (current_time - play_start_time) / 60
            total_play_time[player_out] = total_play_time.get(player_out, 0) + minutes_played
            print(f"🕒 {player_out} שיחק {minutes_played:.2f} דקות")  # ✅ הדפס דקות משחק של שחקן מוחלף


        # Add the substituted-in player to active players
        active_players[team][player_in] = current_time

    # Add remaining play times for players still on the court
    for team, players in active_players.items():
        for player, play_start_time in players.items():
            #minutes_played = (until_time - play_start_time).total_seconds() / 60
            minutes_played = (until_clock - play_start_time) / 60
            total_play_time[player] = total_play_time.get(player, 0) + minutes_played

    # Convert to a DataFrame for easier analysis
    result = pd.DataFrame(list(total_play_time.items()), columns=['Player', 'MinutesPlayed'])
    result = result.sort_values(by='MinutesPlayed', ascending=False).reset_index(drop=True)

    print("Total play time data:", total_play_time)
    print("⏳ שחקנים שעדיין על המגרש:", active_players)
    
    return result


In [198]:
print(pbp_df.columns)

Index(['GAME_ID', 'EVENTNUM', 'EVENTMSGTYPE', 'EVENTMSGACTIONTYPE', 'PERIOD',
       'WCTIMESTRING', 'PCTIMESTRING', 'HOMEDESCRIPTION', 'NEUTRALDESCRIPTION',
       'VISITORDESCRIPTION', 'SCORE', 'SCOREMARGIN', 'PERSON1TYPE',
       'PLAYER1_ID', 'PLAYER1_NAME', 'PLAYER1_TEAM_ID', 'PLAYER1_TEAM_CITY',
       'PLAYER1_TEAM_NICKNAME', 'PLAYER1_TEAM_ABBREVIATION', 'PERSON2TYPE',
       'PLAYER2_ID', 'PLAYER2_NAME', 'PLAYER2_TEAM_ID', 'PLAYER2_TEAM_CITY',
       'PLAYER2_TEAM_NICKNAME', 'PLAYER2_TEAM_ABBREVIATION', 'PERSON3TYPE',
       'PLAYER3_ID', 'PLAYER3_NAME', 'PLAYER3_TEAM_ID', 'PLAYER3_TEAM_CITY',
       'PLAYER3_TEAM_NICKNAME', 'PLAYER3_TEAM_ABBREVIATION',
       'VIDEO_AVAILABLE_FLAG', 'SECONDS_LEFT_IN_Q', 'ABS_TIME'],
      dtype='object')


In [200]:
ttp = calculate_minutes_played(pbp_df, 720)

🏀 חמישייה פותחת שהוספנו: {('Omer Asik', 'NOP'), ('Tobias Harris', 'ORL'), ('Anthony Davis', 'NOP'), ('Jrue Holiday', 'NOP'), ('Elfrid Payton', 'ORL'), ('Tyreke Evans', 'NOP'), ('Nikola Vučević', 'ORL'), ("Kyle O'Quinn", 'ORL'), ('Eric Gordon', 'NOP'), ('Evan Fournier', 'ORL')}
403
🔄 חילוף: Kyle O'Quinn → Aaron Gordon בקבוצה ORL בזמן 403
🕒 Kyle O'Quinn שיחק 6.72 דקות
429
🔄 חילוף: Omer Asik → Ryan Anderson בקבוצה NOP בזמן 429
🕒 Omer Asik שיחק 7.15 דקות
445
🔄 חילוף: Tyreke Evans → John Salmons בקבוצה NOP בזמן 445
🕒 Tyreke Evans שיחק 7.42 דקות
445
🔄 חילוף: Evan Fournier → Ben Gordon בקבוצה ORL בזמן 445
🕒 Evan Fournier שיחק 7.42 דקות
503
🔄 חילוף: Elfrid Payton → Luke Ridnour בקבוצה ORL בזמן 503
🕒 Elfrid Payton שיחק 8.38 דקות
543
🔄 חילוף: Eric Gordon → Jimmer Fredette בקבוצה NOP בזמן 543
🕒 Eric Gordon שיחק 9.05 דקות
543
🔄 חילוף: Jrue Holiday → Austin Rivers בקבוצה NOP בזמן 543
🕒 Jrue Holiday שיחק 9.05 דקות
543
🔄 חילוף: Nikola Vučević → Dewayne Dedmon בקבוצה ORL בזמן 543
🕒 Nikola Vučević שיחק

In [202]:
print(pbp_df[pbp_df['EVENTMSGTYPE'] == 8][['PLAYER1_NAME', 'PLAYER2_NAME']].head(20))


        PLAYER1_NAME     PLAYER2_NAME
60      Kyle O'Quinn     Aaron Gordon
65         Omer Asik    Ryan Anderson
70      Tyreke Evans     John Salmons
71     Evan Fournier       Ben Gordon
78     Elfrid Payton     Luke Ridnour
88       Eric Gordon  Jimmer Fredette
89      Jrue Holiday    Austin Rivers
90    Nikola Vučević   Dewayne Dedmon
99     Anthony Davis        Omer Asik
107    Tobias Harris     Willie Green
123     John Salmons     Tyreke Evans
143   Dewayne Dedmon   Nikola Vučević
160     Luke Ridnour    Elfrid Payton
161     Willie Green    Evan Fournier
172    Ryan Anderson     Jrue Holiday
173  Jimmer Fredette    Anthony Davis
174    Austin Rivers      Eric Gordon
175       Ben Gordon    Tobias Harris
186     Aaron Gordon     Kyle O'Quinn
219        Omer Asik    Ryan Anderson


In [204]:
print(ttp)

             Player  MinutesPlayed
0     Tobias Harris      10.133333
1         Omer Asik       9.583333
2     Anthony Davis       9.566667
3    Nikola Vučević       9.050000
4       Eric Gordon       9.050000
5      Jrue Holiday       9.050000
6     Elfrid Payton       8.383333
7      Tyreke Evans       8.116667
8     Evan Fournier       7.416667
9      Kyle O'Quinn       6.716667
10     Aaron Gordon       5.283333
11    Ryan Anderson       4.850000
12       Ben Gordon       4.583333
13     John Salmons       3.883333
14     Luke Ridnour       3.616667
15    Austin Rivers       2.950000
16  Jimmer Fredette       2.950000
17   Dewayne Dedmon       2.950000
18     Willie Green       1.866667


In [206]:
print(pbp_df['EVENTMSGTYPE'].unique())

[12 10  2  4  1  5  6  3  9  8 18 13]


In [208]:
print(pbp_df[pbp_df['EVENTMSGTYPE'] == 8][['PLAYER1_NAME', 'PLAYER2_NAME', 'PERIOD', 'ABS_TIME']].head(10))


       PLAYER1_NAME     PLAYER2_NAME  PERIOD  ABS_TIME
60     Kyle O'Quinn     Aaron Gordon       1       403
65        Omer Asik    Ryan Anderson       1       429
70     Tyreke Evans     John Salmons       1       445
71    Evan Fournier       Ben Gordon       1       445
78    Elfrid Payton     Luke Ridnour       1       503
88      Eric Gordon  Jimmer Fredette       1       543
89     Jrue Holiday    Austin Rivers       1       543
90   Nikola Vučević   Dewayne Dedmon       1       543
99    Anthony Davis        Omer Asik       1       574
107   Tobias Harris     Willie Green       1       608


In [210]:
print(ttp)


             Player  MinutesPlayed
0     Tobias Harris      10.133333
1         Omer Asik       9.583333
2     Anthony Davis       9.566667
3    Nikola Vučević       9.050000
4       Eric Gordon       9.050000
5      Jrue Holiday       9.050000
6     Elfrid Payton       8.383333
7      Tyreke Evans       8.116667
8     Evan Fournier       7.416667
9      Kyle O'Quinn       6.716667
10     Aaron Gordon       5.283333
11    Ryan Anderson       4.850000
12       Ben Gordon       4.583333
13     John Salmons       3.883333
14     Luke Ridnour       3.616667
15    Austin Rivers       2.950000
16  Jimmer Fredette       2.950000
17   Dewayne Dedmon       2.950000
18     Willie Green       1.866667


In [212]:
print(ttp['Player'].unique())


['Tobias Harris' 'Omer Asik' 'Anthony Davis' 'Nikola Vučević'
 'Eric Gordon' 'Jrue Holiday' 'Elfrid Payton' 'Tyreke Evans'
 'Evan Fournier' "Kyle O'Quinn" 'Aaron Gordon' 'Ryan Anderson'
 'Ben Gordon' 'John Salmons' 'Luke Ridnour' 'Austin Rivers'
 'Jimmer Fredette' 'Dewayne Dedmon' 'Willie Green']
